In [1]:
"""
코랩용 DPO (Direct Preference Optimization) DataSet 생성 스크립트
./random_persona_campaign.csv 파일을 토대로 각 입력 데이터에 맞는 CRM 예시를 생성해
/content/drive/MyDrive/멋사/dpo_dataset/에 저장
"""

'\n코랩용 DPO (Direct Preference Optimization) DataSet 생성 스크립트\n./random_persona_campaign.csv 파일을 토대로 각 입력 데이터에 맞는 CRM 예시를 생성해\n/content/drive/MyDrive/멋사/dpo_dataset/에 저장\n'

In [1]:
import torch
torch.cuda.is_available()

# from google.colab import drive
# drive.mount('/content/drive')

import os
print(os.getcwd())
print(os.listdir())

/content
['.config', '.env', 'drive', '.ipynb_checkpoints', 'AmoRe_crm_generator', 'sample_data']


In [ ]:
!pip install datasets peft trl bitsandbytes accelerate
!pip install -U transformers
!pip show transformers

In [3]:
!git clone https://github.com/jjjh02/AmoRe_crm_generator.git
%cd AmoRe_crm_generator
!git checkout jinhyeok

Cloning into 'AmoRe_crm_generator'...
remote: Enumerating objects: 108, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 108 (delta 46), reused 83 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (108/108), 1.86 MiB | 12.30 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/content/AmoRe_crm_generator
Branch 'jinhyeok' set up to track remote branch 'jinhyeok' from 'origin'.
Switched to a new branch 'jinhyeok'


In [3]:
# !git branch
os.chdir("/content/AmoRe_crm_generator/finetuning")
print(os.getcwd())

/content/AmoRe_crm_generator/finetuning


In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
import argparse
import csv
import inspect
import json
import os
import re
import sys
import tempfile
import time
import urllib.error
import urllib.request
from contextlib import redirect_stdout
from io import StringIO


BASE_DIR = os.getcwd()
DEFAULT_CSV = os.path.join(BASE_DIR, "random_persona_campaign.csv")
# DEFAULT_OUTPUT = os.path.join(BASE_DIR, "finetuning_data_dpo", "cycle_01_v2.json")
DEFAULT_OUTPUT = "/content/drive/MyDrive/LikeLion/dataset_dpo/cycle_01_v3.json"

# 데이터셋 생성 시에는 어댑터 사용 안함 (베이스 모델로 생성)
# 추론 시에만 어댑터 적용 (test.ipynb 등에서)
ADAPTER_PATH = None  # "/content/drive/MyDrive/멋사/adapters_dpo/adapters_dpo_2"
_ADAPTER_PATCHED = False  # 어댑터 미적용 상태

SRC_DIR = os.path.abspath(os.path.join(BASE_DIR, "..", "src"))

PIPELINE_MODULE = None
PIPELINE_CONTEXT = None
PIPELINE_DEFAULTS = {
    "qwen_model": "Qwen/Qwen2.5-1.5B-Instruct",
    "exa_model": "LGAI-EXAONE/EXAONE-4.0-1.2B",
    "top_k": 1,
    "vibe": "2030",  # 타겟 연령대 바이브 (2030 or 4060)
    "use_gate": False,  # DPO 데이터셋 생성 시 GATE 비활성화 (다양한 품질의 후보 필요)
    "gate_max_retries": 3,  # GATE 최대 재시도 횟수
    "style_index": 0,  # 기본 스타일 인덱스
    "is_event": False,  # 기본 이벤트 여부
}

ENABLE_PIPELINE_CACHE = True
ENABLE_BATCH = True

# GPT API 호출 설정
GPT_REQUEST_TIMEOUT = 90  # 30초 -> 90초로 증가
GPT_MAX_RETRIES = 3
GPT_BACKOFF_SECONDS = 5

TIMING_AVG_WINDOW = 10
_TIMING_SUM = {
    "load": 0.0,
    "qwen": 0.0,
    "rag": 0.0,
    "exaone": 0.0,
    "total": 0.0,
}
_TIMING_COUNT = 0

def _log_timing(timing):
    global _TIMING_COUNT
    _TIMING_COUNT += 1
    for key in _TIMING_SUM:
        _TIMING_SUM[key] += float(timing.get(key, 0.0))
    _log(
        "[Timing] "
        f"load={timing.get('load', 0.0):.2f}s "
        f"qwen={timing.get('qwen', 0.0):.2f}s "
        f"rag={timing.get('rag', 0.0):.2f}s "
        f"exaone={timing.get('exaone', 0.0):.2f}s "
        f"total={timing.get('total', 0.0):.2f}s"
    )
    if _TIMING_COUNT % TIMING_AVG_WINDOW == 0:
        avg = {k: _TIMING_SUM[k] / TIMING_AVG_WINDOW for k in _TIMING_SUM}
        _log(
            "[TimingAvg] "
            f"n={TIMING_AVG_WINDOW} "
            f"load={avg['load']:.2f}s "
            f"qwen={avg['qwen']:.2f}s "
            f"rag={avg['rag']:.2f}s "
            f"exaone={avg['exaone']:.2f}s "
            f"total={avg['total']:.2f}s"
        )
        for key in _TIMING_SUM:
            _TIMING_SUM[key] = 0.0

def _log(message):
    print(message)


def _import_pipeline_main():
    global PIPELINE_MODULE
    if SRC_DIR not in sys.path:
        sys.path.insert(0, SRC_DIR)
    try:
        import run_qwen_exaone_pipeline as pipeline_module
        pipeline_main = pipeline_module.main
    except Exception as exc:
        raise ImportError(
            "Failed to import main from ../src/run_qwen_exaone_pipeline.py"
        ) from exc
    _apply_exaone_adapter()
    PIPELINE_MODULE = pipeline_module
    return pipeline_main


def _get_pipeline_context():
    global PIPELINE_CONTEXT
    if PIPELINE_CONTEXT is not None:
        return PIPELINE_CONTEXT
    if PIPELINE_MODULE is None:
        _import_pipeline_main()
    if PIPELINE_MODULE is not None:
        if hasattr(PIPELINE_MODULE, "_set_cache_enabled"):
            PIPELINE_MODULE._set_cache_enabled(ENABLE_PIPELINE_CACHE)
        else:
            try:
                PIPELINE_MODULE.CACHE_ENABLED = ENABLE_PIPELINE_CACHE
            except Exception:
                pass
    base = os.path.dirname(os.path.dirname(os.path.abspath(PIPELINE_MODULE.__file__)))
    if not ENABLE_PIPELINE_CACHE:
        return {
            "base": base,
            "data": None,
            "q_generator": None,
            "exa_generator": None,
        }
    data = None
    q_generator = None
    exa_generator = None
    try:
        data = PIPELINE_MODULE._load_data(base)
    except Exception:
        data = None
    try:
        q_generator = PIPELINE_MODULE._get_qwen_generator(PIPELINE_DEFAULTS["qwen_model"])
    except Exception:
        q_generator = None
    try:
        exa_generator = PIPELINE_MODULE._get_exaone_generator(PIPELINE_DEFAULTS["exa_model"])
    except Exception:
        exa_generator = None
    PIPELINE_CONTEXT = {
        "base": base,
        "data": data,
        "q_generator": q_generator,
        "exa_generator": exa_generator,
    }
    return PIPELINE_CONTEXT


def _apply_exaone_adapter():
    """어댑터 적용 함수 (데이터셋 생성 시에는 ADAPTER_PATH=None으로 스킵)"""
    global _ADAPTER_PATCHED
    if _ADAPTER_PATCHED:
        return
    if not ADAPTER_PATH:
        _log("[Adapter] 어댑터 비활성화 (베이스 모델 사용)")
        return
    if not os.path.exists(ADAPTER_PATH):
        raise FileNotFoundError(f"Adapter not found: {ADAPTER_PATH}")

    import run_qwen_exaone_pipeline as pipeline_module
    import tone_correction
    try:
        from peft import PeftModel
    except ImportError as exc:
        raise RuntimeError("peft is required to load adapters.") from exc

    class PatchedExaoneToneCorrector(tone_correction.ExaoneToneCorrector):
        _cache = {}

        def __init__(self, model_name="LGAI-EXAONE/EXAONE-4.0-1.2B", use_cache=True):
            cache_allowed = use_cache and ENABLE_PIPELINE_CACHE
            key = (model_name, ADAPTER_PATH)
            cached = self._cache.get(key) if cache_allowed else None
            if cached:
                self.device = cached["device"]
                self.model_name = model_name
                self.tokenizer = cached["tokenizer"]
                self.model = cached["model"]
                return
            super().__init__(model_name=model_name, use_cache=use_cache)
            self.model = PeftModel.from_pretrained(self.model, ADAPTER_PATH)
            try:
                self.model.eval()
            except Exception:
                pass
            if cache_allowed:
                self._cache[key] = {
                    "device": self.device,
                    "tokenizer": self.tokenizer,
                    "model": self.model,
                }

    pipeline_module.ExaoneToneCorrector = PatchedExaoneToneCorrector
    _ADAPTER_PATCHED = True
    _log(f"[Adapter] Loaded: {ADAPTER_PATH}")

def _parse_bool(value):
    if isinstance(value, bool):
        return value
    if value is None:
        return False
    if isinstance(value, (int, float)):
        return bool(value)
    text = str(value).strip().lower()
    return text in {"1", "true", "yes", "y", "t"}


def _load_pairs(csv_path):
    """CSV 파일에서 파이프라인 입력 데이터를 로드합니다.
    
    CSV 컬럼 형식:
    - persona_id: 페르소나 인덱스 → pipeline persona
    - persona: 페르소나 이름 (참조용)
    - tone: 바이브 (2030/4060) → pipeline vibe
    - purpose_id: 목적 인덱스 → pipeline stage_index
    - purpose: 목적명 (참조용)
    - brand: 브랜드명 → pipeline brand
    - product: 제품명 → pipeline product
    - style_index: 스타일 인덱스 (0~5) → pipeline style_index
    - is_event: 이벤트 여부 (0/1) → pipeline is_event
    - title, body, cta: 참조용 데이터
    """
    with open(csv_path, "r", newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if not row:
                continue
            
            # 컬럼 파싱
            persona_id_raw = row.get("persona_id", "").strip()
            brand_raw = row.get("brand", "").strip()
            product_raw = row.get("product", "").strip()
            purpose_id_raw = row.get("purpose_id", "").strip()
            tone_raw = row.get("tone", "").strip()
            style_index_raw = row.get("style_index", "").strip()
            is_event_raw = row.get("is_event", "").strip()
            
            # 필수 필드 검증
            if not persona_id_raw or not brand_raw or not product_raw:
                continue
            if not purpose_id_raw:
                continue
            
            try:
                persona = int(persona_id_raw)
                stage_index = int(purpose_id_raw)
            except ValueError:
                continue
            
            # vibe 파싱: tone 컬럼 사용 (2030/4060)
            vibe = tone_raw if tone_raw in ["2030", "4060"] else PIPELINE_DEFAULTS["vibe"]
            
            # style_index 파싱: CSV에서 읽거나 기본값 사용
            try:
                style_index = int(style_index_raw) if style_index_raw else PIPELINE_DEFAULTS["style_index"]
                # 유효 범위 검증 (0~5)
                if not (0 <= style_index <= 5):
                    style_index = PIPELINE_DEFAULTS["style_index"]
            except ValueError:
                style_index = PIPELINE_DEFAULTS["style_index"]
            
            # is_event 파싱: CSV에서 읽거나 기본값 사용
            is_event = _parse_bool(is_event_raw) if is_event_raw else PIPELINE_DEFAULTS["is_event"]
            
            yield {
                "persona": persona,
                "brand": brand_raw,
                "product": product_raw,
                "stage_index": stage_index,
                "style_index": style_index,
                "is_event": is_event,
                "vibe": vibe,
            }


def _format_prompt(summarization):
    if summarization is None:
        return ""
    if isinstance(summarization, str):
        return summarization.strip()
    return json.dumps(summarization, ensure_ascii=False, indent=2)


def _format_event(selected_event):
    if selected_event in (None, "", {}):
        return "없음"
    if isinstance(selected_event, dict):
        for key in ("title", "name", "event_name", "event"):
            if selected_event.get(key):
                return str(selected_event.get(key))
        return json.dumps(selected_event, ensure_ascii=False)
    return str(selected_event)


def _format_price(price):
    if price in (None, ""):
        return ""
    if isinstance(price, (int, float)):
        return f"{int(price):,}원"
    text = str(price).strip()
    if not text:
        return ""
    if "원" in text:
        return text
    if text.replace(",", "").isdigit():
        return f"{int(text.replace(',', '')):,}원"
    return text


def _format_persona(persona_profile):
    if not isinstance(persona_profile, dict):
        return str(persona_profile or "")
    name = persona_profile.get("name", "")
    extras = []
    value_focus = persona_profile.get("value_focus")
    skin_type = persona_profile.get("skin_type")
    traits = persona_profile.get("traits")
    shopping_style = persona_profile.get("shopping_style")
    if value_focus:
        extras.append(str(value_focus))
    if skin_type:
        extras.append(str(skin_type))
    if traits:
        if isinstance(traits, list):
            extras.append(", ".join([str(t) for t in traits if t]))
        else:
            extras.append(str(traits))
    if shopping_style:
        extras.append(str(shopping_style))
    extra_text = ", ".join([e for e in extras if e])
    if name and extra_text:
        return f"{name} ({extra_text})"
    return name or extra_text


def _build_prompt_text(meta, fallback_text):
    persona = _format_persona(meta.get("persona_profile") if isinstance(meta, dict) else None)
    stage = ""
    if isinstance(meta, dict):
        stage = meta.get("stage_name") or meta.get("stage_kr") or ""
    brand = meta.get("brand") if isinstance(meta, dict) else ""
    product_basic = meta.get("product_basic") if isinstance(meta, dict) else None
    product_name = ""
    price = ""
    if isinstance(product_basic, dict):
        product_name = product_basic.get("name", "") or ""
        price = _format_price(product_basic.get("price"))
    product_query = meta.get("product_query") if isinstance(meta, dict) else ""
    if not product_name:
        product_name = product_query or ""

    event_text = _format_event(meta.get("selected_event") if isinstance(meta, dict) else None)

    # 추가: vibe, vibe_text, selected_context_example 정보
    vibe = meta.get("vibe", "") if isinstance(meta, dict) else ""
    vibe_text = meta.get("vibe_text", "") if isinstance(meta, dict) else ""
    context_example = meta.get("selected_context_example", "") if isinstance(meta, dict) else ""

    lines = ["[컨텍스트]"]
    if persona:
        lines.append(f"- Persona: {persona}")
    if stage:
        lines.append(f"- Stage: {stage}")
    if brand or product_name:
        lines.append(f"- Brand/Product: {brand} / {product_name}".strip())
    if price:
        lines.append(f"- Price: {price}")
    lines.append(f"- Event: {event_text}")
    # 추가: vibe 및 context_example 정보
    if vibe:
        lines.append(f"- Vibe: {vibe}")
    if vibe_text:
        lines.append(f"- Tone Guide: {vibe_text[:100]}..." if len(vibe_text) > 100 else f"- Tone Guide: {vibe_text}")
    if context_example:
        lines.append(f"- Context Example: {context_example}")

    prompt = "\n".join(lines).strip()
    if prompt:
        return prompt
    return _format_prompt(fallback_text)


def _candidate_text(candidate):
    if isinstance(candidate, dict):
        return (
            candidate.get("text")
            or candidate.get("crm_message")
            or candidate.get("message")
            or candidate.get("content")
        )
    return str(candidate)


def _candidate_meta(candidate):
    if isinstance(candidate, dict):
        return candidate.get("meta") or {}
    return {}


def _normalize_candidates(raw):
    if raw is None:
        return []
    if isinstance(raw, dict):
        if "candidates" in raw:
            raw = raw["candidates"]
        elif "messages" in raw:
            raw = raw["messages"]
        elif "crm_messages" in raw:
            raw = raw["crm_messages"]
        elif "crm_message" in raw and isinstance(raw["crm_message"], list):
            raw = raw["crm_message"]
        else:
            raw = [raw]
    if not isinstance(raw, list):
        raw = [raw]

    normalized = []
    for idx, item in enumerate(raw):
        text = _candidate_text(item)
        if not text:
            continue
        normalized.append(
            {
                "response_id": idx,
                "text": text,
                "meta": _candidate_meta(item),
            }
        )
    return normalized


def _dedupe_candidates(items):
    seen = set()
    result = []
    for item in items:
        text = _candidate_text(item)
        if not text:
            continue
        key = text.strip()
        if not key or key in seen:
            continue
        seen.add(key)
        result.append(item)
    return result


def _extract_response_text(data):
    if isinstance(data, dict):
        output_text = data.get("output_text")
        if isinstance(output_text, str) and output_text.strip():
            return output_text.strip()

        output = data.get("output")
        if isinstance(output, list):
            parts = []
            for item in output:
                if not isinstance(item, dict):
                    continue
                content = item.get("content", [])
                if isinstance(content, list):
                    for block in content:
                        if isinstance(block, dict) and isinstance(block.get("text"), str):
                            parts.append(block["text"])
                        elif isinstance(block, str):
                            parts.append(block)
                elif isinstance(content, str):
                    parts.append(content)
            if parts:
                return "".join(parts).strip()

    raise ValueError(f"Invalid evaluator response: {data}")


def _format_meta(meta):
    if not isinstance(meta, dict):
        return str(meta)

    lines = []
    persona_profile = meta.get("persona_profile")
    if persona_profile is not None:
        lines.append(f"persona_profile: {json.dumps(persona_profile, ensure_ascii=False)}")
    brand = meta.get("brand")
    if brand:
        lines.append(f"brand: {brand}")
    stage_kr = meta.get("stage_kr")
    if stage_kr:
        lines.append(f"stage_kr: {stage_kr}")
    objective = meta.get("objective")
    if objective:
        lines.append(f"objective: {objective}")
    target_state = meta.get("target_state")
    if target_state:
        lines.append(f"target_state: {target_state}")
    style_templates = meta.get("style_templates")
    if style_templates:
        if isinstance(style_templates, list):
            lines.append("style_templates:")
            for item in style_templates:
                lines.append(f"- {item}")
        else:
            lines.append(f"style_templates: {style_templates}")
    selected_event = meta.get("selected_event")
    if selected_event is not None:
        if isinstance(selected_event, (dict, list)):
            event_text = json.dumps(selected_event, ensure_ascii=False)
        else:
            event_text = str(selected_event)
        lines.append(f"selected_event: {event_text}")

    return "\n".join(lines) if lines else "(context unavailable)"


def _call_gpt(prompt_text, candidates):
    """GPT를 호출하여 CRM 메시지 후보들을 평가합니다.

    평가 기준은 tone_correction.py의 프롬프트 형식 규칙을 기반으로 합니다.
    """
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY is not set.")

    candidate_lines = []
    for idx, candidate in enumerate(candidates):
        text = candidate.get("text", "")
        candidate_lines.append(f"[{idx}]\n{text}")
    candidate_block = "\n\n".join(candidate_lines)

    # tone_correction.py 기반 심사 기준으로 업데이트
    system_prompt = (
        "너는 CRM 메시지의 품질을 평가하는 심사자다.\n"
        "목표는 아래 규칙을 가장 잘 준수하는 메시지를 고르는 것이다.\n"
        "\n"
        "## 필수 출력 형식 규칙\n"
        "1) 제목은 한 줄 분량(20~25자)으로 간결하게 작성되어야 한다.\n"
        "2) 본문은 두 줄~세 줄 분량(80~120자)으로 작성되어야 한다.\n"
        "3) 본문의 마지막 한 줄은 반드시 CTA(Call-to-Action) 문장이어야 한다.\n"
        "4) 출력 형식은 반드시 '[제목]'과 '[본문]' 레이블을 사용해야 한다.\n"
        "\n"
        "## 금지 요소 (감점 대상)\n"
        "- 영어 사용 금지: 브랜드/제품 고유명 제외, 한국어로만 작성\n"
        "  (금지 예시: everyday, daily, routine, essential, ultimate, available 등)\n"
        "- 페르소나/개인정보 직접 호명 금지\n"
        "  (금지 예시: CL_01님, Budget_Seeker님, user_name 등)\n"
        "- 메타 표현/특수문자 금지: { } ( ) * : ; # > < % _ 등\n"
        "- JSON/코드 조각 금지: ```json, ### 등\n"
        "- 과도한 이모지/구분선 금지\n"
        "\n"
        "## 표현 수위 가이드\n"
        "- 허용: '도움을 줄 수 있어요', '편안하게 느껴질 수 있어요', '부담 없이 사용하기 좋아요'\n"
        "- 지양: '완벽 개선', '즉시 효과', '100% 보장', '치료' 등 의학적/단정 표현\n"
        "\n"
        "## 평가 우선순위\n"
        "1. 금지 요소 위반 여부 (위반 시 큰 감점)\n"
        "2. 출력 형식 준수 여부\n"
        "3. 제목/본문 글자 수 적정성\n"
        "4. CTA 포함 여부\n"
        "5. 자연스러운 한국어 표현\n"
        "\n"
        "출력은 반드시 JSON 하나로만 해라.\n"
        "형식: {\"best_index\":0,\"reason_best\":\"...\",\"reason_others\":[{\"index\":1,\"reason\":\"...\"}]}\n"
        "reason은 위 규칙 중 어떤 부분을 잘 지켰거나 위반했는지 1~2문장으로 간단하게 작성하라.\n"
    )
    user_prompt = (
        "컨텍스트\n"
        f"{prompt_text}\n\n"
        "후보\n"
        f"{candidate_block}\n\n"
        "JSON만 출력하라."
    )

    payload = {
        "model": "gpt-5-nano",
        "input": [
            {
                "role": "system",
                "content": [{"type": "input_text", "text": system_prompt}],
            },
            {
                "role": "user",
                "content": [{"type": "input_text", "text": user_prompt}],
            },
        ],
    }

    request = urllib.request.Request(
        "https://api.openai.com/v1/responses",
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        method="POST",
    )

    # 재시도 로직 추가
    last_error = None
    data = None
    for attempt in range(1, GPT_MAX_RETRIES + 1):
        try:
            with urllib.request.urlopen(request, timeout=GPT_REQUEST_TIMEOUT) as response:
                data = json.loads(response.read().decode("utf-8"))
            break  # 성공 시 루프 탈출
        except urllib.error.HTTPError as exc:
            body = exc.read().decode("utf-8", errors="replace")
            if exc.code in {408, 429, 500, 502, 503, 504} and attempt < GPT_MAX_RETRIES:
                _log(f"[Retry] HTTP {exc.code} attempt {attempt}/{GPT_MAX_RETRIES}")
                time.sleep(GPT_BACKOFF_SECONDS * attempt)
                last_error = exc
                continue
            raise RuntimeError(f"OpenAI API error {exc.code}: {body}") from exc
        except (TimeoutError, urllib.error.URLError, OSError) as exc:
            if attempt < GPT_MAX_RETRIES:
                _log(f"[Retry] Timeout/URLError attempt {attempt}/{GPT_MAX_RETRIES}: {exc}")
                time.sleep(GPT_BACKOFF_SECONDS * attempt)
                last_error = exc
                continue
            raise RuntimeError(f"OpenAI API request timed out after {GPT_MAX_RETRIES} attempts.") from exc
    
    if data is None:
        raise RuntimeError("OpenAI API request failed after all retries.") from last_error

    content = _extract_response_text(data)
    text = content.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z0-9]*", "", text)
        text = re.sub(r"```$", "", text).strip()

    parsed = None
    match_json = re.search(r"\{.*\}", text, re.DOTALL)
    if match_json:
        try:
            parsed = json.loads(match_json.group(0))
        except json.JSONDecodeError:
            parsed = None

    best_index = None
    reason_best = ""
    reason_others = []
    if isinstance(parsed, dict):
        best_index = parsed.get("best_index")
        if best_index is None:
            best_index = parsed.get("index") or parsed.get("choice")
        reason_best = parsed.get("reason_best") or parsed.get("reason") or ""
        reason_others = parsed.get("reason_others") or parsed.get("reasons_others") or []
    if best_index is None:
        match = re.search(r"-?\d+", text)
        if not match:
            raise ValueError(f"Invalid evaluator response: {content}")
        best_index = int(match.group(0))
    if isinstance(best_index, str) and best_index.strip().isdigit():
        best_index = int(best_index.strip())
    if best_index is None or best_index < 0 or best_index >= len(candidates):
        raise ValueError(f"Evaluator index out of range: {best_index}")
    if isinstance(reason_others, dict):
        reason_others = [
            {"index": k, "reason": v} for k, v in reason_others.items()
        ]
    return best_index, reason_best, reason_others

def _load_existing_records(path):
    if not os.path.exists(path):
        return []
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError(f"Expected a list in {path}")
    return data


def _save_records(path, records):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)


def _extract_pipeline_output(result):
    """파이프라인 출력에서 필요한 정보를 추출합니다.

    업데이트: vibe, vibe_text, selected_context_example, GATE 관련 정보 추가
    """
    if isinstance(result, dict):
        if "qwen" in result or "exaone" in result:
            qwen = result.get("qwen", {}) or {}
            exaone = result.get("exaone", {}) or {}
            summarization = qwen.get("draft") or qwen.get("qwen_draft")

            # GATE 검증 결과에 따라 적절한 출력 선택
            gate_passed = exaone.get("gate_passed")
            if gate_passed:
                # GATE 통과 시 formatted 결과 사용
                crm_message = exaone.get("result_formatted") or exaone.get("result_raw") or exaone.get("crm_message")
            else:
                # GATE 미통과 또는 비활성화 시 raw 결과 사용
                crm_message = exaone.get("result_raw") or exaone.get("crm_message")

            meta = {
                "persona_profile": result.get("persona_profile"),
                "brand": result.get("brand"),
                "product_basic": result.get("product_basic"),
                "product_query": result.get("product_query"),
                "stage_name": result.get("stage_name"),
                "stage_kr": result.get("stage_kr"),
                "objective": result.get("objective"),
                "target_state": result.get("target_state"),
                "style_templates": result.get("style_templates"),
                "selected_event": result.get("selected_event"),
                # 추가: 새로운 필드들
                "vibe": result.get("vibe"),
                "vibe_text": result.get("vibe_text"),
                "selected_context_example": result.get("selected_context_example"),
                # GATE 관련 정보
                "gate_enabled": exaone.get("gate_enabled"),
                "gate_passed": exaone.get("gate_passed"),
                "gate_attempts": exaone.get("gate_attempts"),
                # 제목/본문 분리된 결과
                "result_title": exaone.get("result_title"),
                "result_body": exaone.get("result_body"),
            }
            return summarization, crm_message, meta
        summarization = result.get("summarization")
        crm_message = result.get("crm_message")
        return summarization, crm_message, {}
    if isinstance(result, (list, tuple)) and len(result) >= 2:
        summarization, crm_message = result[0], result[1]
        return summarization, crm_message, {}
    if isinstance(result, str):
        try:
            parsed = json.loads(result)
        except json.JSONDecodeError as exc:
            raise ValueError(f"Unexpected pipeline output: {result}") from exc
        return _extract_pipeline_output(parsed)
    raise ValueError(f"Unexpected pipeline output: {result}")


def _parse_stdout_payload(stdout_text):
    for line in reversed(stdout_text.splitlines()):
        line = line.strip()
        if not line:
            continue
        if not line.startswith("{") or not line.endswith("}"):
            continue
        try:
            return json.loads(line)
        except json.JSONDecodeError:
            continue
    raise ValueError("Pipeline did not return a usable payload.")


def _run_pipeline_via_argv(pipeline_main, row):
    argv = [
        "run_qwen_exaone_pipeline.py",
        "--persona",
        str(row["persona"]),
        "--brand",
        row["brand"],
        "--product",
        row["product"],
        "--stage_index",
        str(row["stage_index"]),
        "--style_index",
        str(row["style_index"]),
        "--is_event",
        "1" if row.get("is_event", False) else "0",
        # 추가: 새로운 파라미터들
        "--vibe",
        row.get("vibe", PIPELINE_DEFAULTS["vibe"]),
        "--use_gate",
        "1" if row.get("use_gate", PIPELINE_DEFAULTS["use_gate"]) else "0",
        "--gate_max_retries",
        str(row.get("gate_max_retries", PIPELINE_DEFAULTS["gate_max_retries"])),
    ]
    if not ENABLE_PIPELINE_CACHE:
        argv.append("--disable_cache")
    buf = StringIO()
    old_argv = sys.argv
    try:
        sys.argv = argv
        with redirect_stdout(buf):
            result = pipeline_main()
    finally:
        sys.argv = old_argv
    if result is not None:
        return result
    return _parse_stdout_payload(buf.getvalue())


def _run_pipeline(pipeline_main, row):
    params = {
        "persona": row["persona"],
        "brand": row["brand"],
        "product": row["product"],
        "stage_index": row["stage_index"],
        "style_index": row["style_index"],
        "is_event": row.get("is_event", False),
        # 추가: 새로운 파라미터들
        "vibe": row.get("vibe", PIPELINE_DEFAULTS["vibe"]),
        "use_gate": row.get("use_gate", PIPELINE_DEFAULTS["use_gate"]),
        "gate_max_retries": row.get("gate_max_retries", PIPELINE_DEFAULTS["gate_max_retries"]),
    }
    if PIPELINE_MODULE is None:
        _import_pipeline_main()
    if PIPELINE_MODULE is not None:
        if hasattr(PIPELINE_MODULE, "_set_cache_enabled"):
            PIPELINE_MODULE._set_cache_enabled(ENABLE_PIPELINE_CACHE)
        else:
            try:
                PIPELINE_MODULE.CACHE_ENABLED = ENABLE_PIPELINE_CACHE
            except Exception:
                pass
    if PIPELINE_MODULE is not None and hasattr(PIPELINE_MODULE, "_run_pipeline"):
        ctx = _get_pipeline_context()
        args = argparse.Namespace(
            persona=params["persona"],
            brand=params["brand"],
            product=params["product"],
            stage_index=params["stage_index"],
            style_index=params["style_index"],
            is_event=1 if params["is_event"] else 0,
            top_k=PIPELINE_DEFAULTS["top_k"],
            qwen_model=PIPELINE_DEFAULTS["qwen_model"],
            exa_model=PIPELINE_DEFAULTS["exa_model"],
            out_path=None,
            batch_json=None,
            disable_cache=not ENABLE_PIPELINE_CACHE,
            # 추가: 새로운 파라미터들
            vibe=params["vibe"],
            use_gate=params["use_gate"],
            gate_max_retries=params["gate_max_retries"],
        )
        return PIPELINE_MODULE._run_pipeline(
            args,
            data=ctx.get("data"),
            q_generator=ctx.get("q_generator"),
            exa_generator=ctx.get("exa_generator"),
        )

    try:
        sig = inspect.signature(pipeline_main)
    except (TypeError, ValueError):
        sig = None

    if sig is not None and len(sig.parameters) == 0:
        return _run_pipeline_via_argv(pipeline_main, row)

    try:
        return pipeline_main(**params)
    except TypeError:
        pass

    try:
        if sig is not None and len(sig.parameters) == 1:
            return pipeline_main(row)
    except TypeError:
        pass

    ordered = [
        params["persona"],
        params["brand"],
        params["product"],
        params["stage_index"],
        params["style_index"],
        params["is_event"],
    ]
    return pipeline_main(*ordered)


def _run_pipeline_batch(pipeline_main, rows):
    if not ENABLE_BATCH:
        outputs = []
        for row in rows:
            outputs.append(_run_pipeline(pipeline_main, row))
        return outputs
    if PIPELINE_MODULE is None:
        _import_pipeline_main()
    if PIPELINE_MODULE is not None:
        if hasattr(PIPELINE_MODULE, "_set_cache_enabled"):
            PIPELINE_MODULE._set_cache_enabled(ENABLE_PIPELINE_CACHE)
        else:
            try:
                PIPELINE_MODULE.CACHE_ENABLED = ENABLE_PIPELINE_CACHE
            except Exception:
                pass
    if PIPELINE_MODULE is not None and hasattr(PIPELINE_MODULE, "main"):
        payload = []
        for row in rows:
            payload.append({
                "persona": row["persona"],
                "brand": row["brand"],
                "product": row["product"],
                "stage_index": row["stage_index"],
                "style_index": row["style_index"],
                "is_event": row.get("is_event", False),
                # 추가: 새로운 파라미터들
                "vibe": row.get("vibe", PIPELINE_DEFAULTS["vibe"]),
                "use_gate": row.get("use_gate", PIPELINE_DEFAULTS["use_gate"]),
                "gate_max_retries": row.get("gate_max_retries", PIPELINE_DEFAULTS["gate_max_retries"]),
            })
        with tempfile.NamedTemporaryFile("w", delete=False, suffix=".json", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False)
            tmp_path = f.name
        argv = [
            "run_qwen_exaone_pipeline.py",
            "--batch_json",
            tmp_path,
            "--qwen_model",
            PIPELINE_DEFAULTS["qwen_model"],
            "--exa_model",
            PIPELINE_DEFAULTS["exa_model"],
            "--top_k",
            str(PIPELINE_DEFAULTS["top_k"]),
        ]
        if not ENABLE_PIPELINE_CACHE:
            argv.append("--disable_cache")
        buf = StringIO()
        old_argv = sys.argv
        try:
            sys.argv = argv
            with redirect_stdout(buf):
                result = pipeline_main()
        finally:
            sys.argv = old_argv
            try:
                os.remove(tmp_path)
            except Exception:
                pass
        if result is not None:
            return result
        return _parse_stdout_payload(buf.getvalue())
    outputs = []
    for row in rows:
        outputs.append(_run_pipeline(pipeline_main, row))
    return outputs


def _collect_candidates(inference_pipeline, row, num_candidates):
    summarization = None
    candidates = []
    summary_mismatch = False
    batch_rows = [row for _ in range(num_candidates)]
    results = _run_pipeline_batch(inference_pipeline, batch_rows)
    if not isinstance(results, list):
        results = [results]
    for attempt, result in enumerate(results, start=1):
        _log(f"  [Attempt {attempt}/{len(results)}] Running pipeline")
        timing = result.get("timing") if isinstance(result, dict) else None
        if timing:
            _log_timing(timing)
        s, message, meta = _extract_pipeline_output(result)
        if summarization is None and s:
            summarization = s
        elif s and summarization and s != summarization:
            summary_mismatch = True
        if isinstance(message, str) and message.strip():
            candidates.append({"text": message.strip(), "meta": meta})
    if summary_mismatch:
        _log("  [Warn] Qwen draft differs across candidates; using the first one.")
    return summarization, candidates


def generate_dpo_data(csv_path, output_path, max_rows=None, num_candidates=4):
    inference_pipeline = _import_pipeline_main()
    records = _load_existing_records(output_path)

    PAIRS_PER_ROW = max(1, num_candidates - 1)
    start_row = len(records) // PAIRS_PER_ROW

    _log(f"Loaded existing records: {len(records)}")
    _log(f"Resuming from CSV row index: {start_row}")

    _log(f"CSV: {csv_path}")
    _log(f"Output: {output_path}")
    _log(f"Candidates per row: {num_candidates}")
    _log(f"Loaded existing records: {len(records)}")

    added = 0
    SAVE_EVERY_N_ROWS = 4
    processed_rows = 0
    for idx, row in enumerate(_load_pairs(csv_path)):
        if idx < start_row or idx < 400 or idx > 500:
          continue
        if max_rows is not None and idx >= max_rows:
            break

        _log(
            "[Row {idx}] persona={persona} brand={brand} product={product} "
            "stage_index={stage_index} style_index={style_index} is_event={is_event} vibe={vibe}".format(
                idx=idx,
                persona=row["persona"],
                brand=row["brand"],
                product=row["product"],
                stage_index=row["stage_index"],
                style_index=row["style_index"],
                is_event=row.get("is_event", False),
                vibe=row.get("vibe", PIPELINE_DEFAULTS["vibe"]),
            )
        )

        try:
            summarization, candidates = _collect_candidates(
                inference_pipeline, row, num_candidates
            )
        except Exception as exc:
            _log(f"[Row {idx}] Pipeline error: {exc}")
            continue

        meta_for_prompt = candidates[0].get("meta", {}) if candidates else {}
        prompt_text = _build_prompt_text(meta_for_prompt, summarization)
        if not prompt_text:
            _log(f"[Row {idx}] Empty summarization, skipping")
            continue

        _log(f"[Row {idx}] Raw candidates: {len(candidates)}")
        candidates = _dedupe_candidates(_normalize_candidates(candidates))
        _log(f"[Row {idx}] Deduped candidates: {len(candidates)}")
        if len(candidates) < 2:
            _log(f"[Row {idx}] Not enough candidates, skipping")
            continue

        try:
            best_idx, reason_best, reason_others = _call_gpt(prompt_text, candidates)
            reasons_by_index = {}
            if reason_others:
                for item in reason_others:
                    if not isinstance(item, dict):
                        continue
                    other_idx = item.get("index")
                    if isinstance(other_idx, str) and other_idx.strip().isdigit():
                        other_idx = int(other_idx.strip())
                    if isinstance(other_idx, int):
                        reasons_by_index[other_idx] = item.get("reason") or ""
            if reason_best:
                _log(f"[Row {idx}] Best reason: {reason_best}")
            if reason_others:
                for item in reason_others:
                    if not isinstance(item, dict):
                        continue
                    other_idx = item.get("index")
                    other_reason = item.get("reason") or ""
                    if other_reason:
                        _log(f"[Row {idx}] Other {other_idx}: {other_reason}")
        except Exception as exc:
            _log(f"[Row {idx}] Evaluator error: {exc}")
            continue

        _log(f"[Row {idx}] Best candidate index: {best_idx}")
        best_text = candidates[best_idx]["text"]
        for candidate_idx, candidate in enumerate(candidates):
            if candidate is candidates[best_idx]:
                continue
            rejected_text = candidate["text"]
            if not rejected_text:
                continue
            rejected_reason = reasons_by_index.get(candidate_idx, "")
            records.append(
                {
                    "prompt": prompt_text,
                    "chosen": best_text,
                    "rejected": rejected_text,
                    "best_index": best_idx,
                    "rejected_index": candidate_idx,
                    "reason_best": reason_best,
                    "reason_rejected": rejected_reason,
                }
            )
            added += 1

        # row 1개 처리 완료
        processed_rows += 1

        if processed_rows % SAVE_EVERY_N_ROWS == 0:
          _save_records(output_path, records)
          _log(f"[Checkpoint] Saved after {processed_rows} rows")

    _save_records(output_path, records)
    _log(f"Saved {added} new records to {output_path}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--csv_path", default=DEFAULT_CSV)
    parser.add_argument("--output_path", default=DEFAULT_OUTPUT)
    parser.add_argument("--max_rows", type=int, default=None)
    parser.add_argument("--num_candidates", type=int, default=4)
    args, _ = parser.parse_known_args()

    generate_dpo_data(
        args.csv_path,
        args.output_path,
        args.max_rows,
        args.num_candidates,
    )


if __name__ == "__main__":
    main()

In [1]:
!pip install huggingface-hub

In [7]:
# Push to HuggingFace
import os
from dotenv import load_dotenv
from huggingface_hub import login, create_repo, upload_folder

load_dotenv("../.env")
login(os.getenv("HUGGINGFACE_API_KEY"))

create_repo(
    repo_id="crm-dpo-dataset",
    repo_type="dataset",
    private=False,
    exist_ok=True
)

upload_folder(
    repo_id="jinn33/crm-dpo-dataset",
    folder_path="./finetuning_data_dpo/crm-dpo-dataset/",
    repo_type="dataset",
    commit_message="Initial Commit"
)

CommitInfo(commit_url='https://huggingface.co/datasets/jinn33/crm-dpo-dataset/commit/b618cb9d24b37a17e151356d65282694f5dbdb38', commit_message='Initial Commit', commit_description='', oid='b618cb9d24b37a17e151356d65282694f5dbdb38', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/jinn33/crm-dpo-dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='jinn33/crm-dpo-dataset'), pr_revision=None, pr_num=None)

In [ ]:
# DPO 데이터셋 재구성 - v3
# DPO 데이터셋 중 chosen을 정제한 SFT 데이터셋을 다시 가져와 본래 DPO chosen 치환
import os
import json

DATASET_DIR = "./finetuning_data"

with open(os.path.join(DATASET_DIR, "crm-sft-dataset", "cycle_01_v2.jsonl"), "r") as f:
    sft_data = f.readlines()
    sft_data = [json.loads(s) for s in sft_data]
    print(len(sft_data), type(sft_data))
    print(sft_data[155])

with open(os.path.join(DATASET_DIR, "crm-dpo-dataset", "cycle_01_v2.json"), "r") as f:
    dpo_data = json.load(f)
    print(len(dpo_data), type(dpo_data))
    print(dpo_data[155])

new_dpo_data = [
    {
        "prompt": sft_data[i]["prompt"],
        "chosen": sft_data[i]["chosen"],
        "rejected": dpo_data[i]["rejected"]
    } for i in range(len(sft_data))
]
print(len(new_dpo_data))

with open(os.path.join(DATASET_DIR, "crm-dpo-dataset", "cycle_01_v3.jsonl"), "w") as f:
    for data in new_dpo_data:
        f.write(json.dumps(data, ensure_ascii=False) + "\n")

# print([type(sft_data[i]) for i in range(len(sft_data))])

1248 <class 'list'>
{'prompt': "다음 조건에 맞는 CRM 메시지를 작성하세요. [컨텍스트]\n- Persona: Sensitive_Skin (저자극, 피부 안정, 임상 테스트 여부, 민감성, 트러블, 장벽 손상, 붉은기, 성분 안정성 최우선, 검증된 브랜드 위주, 신규 제품 보수적)\n- Stage: Retention\n- Brand/Product: 라네즈 / 워터뱅크 블루 히알루로닉 모이스춰 크림 20ml\n- Price: 16,560원\n- Event: 없음  [출력 규칙] - 다음 요소는 포함하지 않는다:   1) 영어/한국어의 어색한 혼용 (브랜드/제품 고유명 제외, 예: 'everyday 사용')   2) 페르소나/개인정보의 직접 호명 (예: 'Budget_Seeker님')   3) 과도한 특수문자, 이모지, 구분선  [출력 형식] 제목: 본문: ", 'chosen': '제목: 민감 피부를 위한 라네즈 워터뱅크 블루 히알루로닉 모이스처 크림 - 20ml\n본문: 피부 장벽 강화에 초점을 맞춘 라네즈 워터뱅크 블루 히알루로닉 모이스처 크림은 민감한 피부도 임상 테스트를 거친 안정적인 포뮬러로 붉은기 완화와 지속적인 보습을 제공합니다. 사용 후기에서도 수 시간 동안 유지되는 보습 효과가 확인되었으며, 검증된 브랜드의 신뢰성도 큰 강점입니다. 현재 20ml 용량으로 16,560원에 만나보실 수 있으며, 용량에 부담 없이 바로 사용해 보시길 권합니다.'}
1248 <class 'list'>
{'prompt': '[컨텍스트]\n- Persona: Sensitive_Skin (저자극, 피부 안정, 임상 테스트 여부, 민감성, 트러블, 장벽 손상, 붉은기, 성분 안정성 최우선, 검증된 브랜드 위주, 신규 제품 보수적)\n- Stage: Retention\n- Brand/Product: 라네즈 / 워터뱅크 블루 히알루로닉 모이스춰 크림 20ml\n- Price: 16,560원\n- Event: 없음', 'chosen': '```json\n{

In [ ]:
# DPO 데이터셋 재재구성 - v4
# cycle_01 -> 페르소나 유출
# cycle_02 -> 한/영 혼용
# cycle_03 -> 대괄호 표현

import json
import os
import re
import urllib.error
import urllib.request
import random
import time
import socket
from dotenv import load_dotenv

load_dotenv()

DATASET_DIR = "./finetuning_data"
INPUT_PATH = os.path.join(DATASET_DIR, "crm-sft-dataset", "cycle_01_v2.jsonl")
OUTPUT_PATH = os.path.join(DATASET_DIR, "crm-dpo-dataset", "cycle_01_v4.jsonl")
TARGET_RULE = "persona"  # persona | mixing | brackets
MODEL_NAME = "gpt-5-nano"
MAX_SENTENCES_PER_ROW = None

REQUEST_TIMEOUT = 90
MAX_RETRIES = 3
BACKOFF_SECONDS = 5
PRINT_EVERY = None
MAX_RECORDS = None
MAX_INPUT_LINES = None
MAX_PROMPT_CHARS = None
DEBUG_LOG = False
LOG_RESULTS = True
LOG_RESUME = False



def _extract_persona(prompt_text):
    if not prompt_text:
        return ""
    match = re.search(r"Persona:\s*([^\s\(\n]+)", prompt_text, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    match = re.search(r"페르소나\s*[:：]\s*([^\s\(\n]+)", prompt_text)
    if match:
        return match.group(1).strip()
    return ""


def _build_fixed_prompt(persona):
    persona = persona.strip() if persona else ""
    if persona:
        return f"{persona} 고객을 위한 CRM 메시지를 작성하라."
    return "고객을 위한 CRM 메시지를 작성하라."


def _extract_body(text):
    if not text:
        return ""
    if "본문:" in text:
        body = text.split("본문:", 1)[1]
    else:
        body = re.sub(r"^제목\s*[:：].*?(\n|$)", "", text).strip()
    body = body.replace("\r", " ").replace("\n", " ")
    body = re.sub(r"\s+", " ", body).strip()
    return body


def _split_sentences(text):
    if not text:
        return []
    text = text.strip()
    parts = re.split(r"(?<=[\.!?…])\s+|\n+", text)
    return [p.strip() for p in parts if p and p.strip()]


def _has_korean(text):
    return bool(re.search(r"[가-힣]", text))


def _has_english(text):
    return bool(re.search(r"[A-Za-z]", text))


def _violates_persona(text, persona):
    if not text or not persona:
        return False
    variants = {
        persona,
        persona.replace("_", " "),
        persona.replace("_", ""),
        persona.lower(),
    }
    for variant in variants:
        if not variant:
            continue
        if re.search(re.escape(variant), text, re.IGNORECASE):
            return True
    return False


def _strip_allowed_english(text):
    if not text:
        return ""
    stripped = text
    stripped = re.sub(r"(?i)\bSPF\s*\d*\+?(?=\b|[가-힣ㄱ-ㅎㅏ-ㅣ])", "", stripped)
    stripped = re.sub(r"(?i)\bPA\+{1,}(?=\b|[가-힣ㄱ-ㅎㅏ-ㅣ])", "", stripped)
    stripped = re.sub(r"(?i)\b\d+\s*(ml|kg|oz|mm|cm|mah)(?=\b|[가-힣ㄱ-ㅎㅏ-ㅣ])", "", stripped)
    stripped = re.sub(r"(?i)\b\d+\s*g(?=\b|[가-힣ㄱ-ㅎㅏ-ㅣ])", "", stripped)
    stripped = re.sub(r"(?i)\b(?:ml|kg|oz|mm|cm|mah)(?=\b|[가-힣ㄱ-ㅎㅏ-ㅣ])", "", stripped)
    stripped = re.sub(r"(?i)\bg(?=\b|[가-힣ㄱ-ㅎㅏ-ㅣ])", "", stripped)
    return stripped

def _violates_mixing(text):
    if not _has_korean(text):
        return False
    stripped = _strip_allowed_english(text)
    return _has_english(stripped)


def _violates_brackets(text):
    return "[" in text or "]" in text


def _rule_violation(text, persona, rule):
    if rule == "persona":
        return _violates_persona(text, persona)
    if rule == "mixing":
        return _violates_mixing(text)
    if rule == "brackets":
        return _violates_brackets(text)
    raise ValueError(f"Unknown rule: {rule}")


def _all_violations(text, persona):
    return {
        "persona": _violates_persona(text, persona),
        "mixing": _violates_mixing(text),
        "brackets": _violates_brackets(text),
    }


def _only_target_violation(text, persona, rule):
    flags = _all_violations(text, persona)
    return flags.get(rule) and sum(1 for v in flags.values() if v) == 1


def _no_violations(text, persona):
    flags = _all_violations(text, persona)
    return not any(flags.values())


def _pick_violation_rule():
    r = random.random()
    if r < 0.6:
        return "persona"
    if r < 0.9:
        return "mixing"
    return "brackets"


def _persona_variants(persona):
    if not persona:
        return []
    return [
        persona,
        persona.replace("_", " "),
        persona.replace("_", ""),
        persona.lower(),
    ]


def _fix_persona_local(sentence, persona):
    variants = _persona_variants(persona)
    if not variants:
        return sentence
    if random.random() < 0.7:
        text = sentence
        for v in variants:
            if not v:
                continue
            text = re.sub(rf"{re.escape(v)}\s*님?", "고객님", text, flags=re.IGNORECASE)
        return text

    text = sentence
    for v in variants:
        if not v:
            continue
        text = re.sub(rf"{re.escape(v)}\s*님?\s*,?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def _fix_brackets_local(sentence):
    text = re.sub(r"\[[^\]]*\]", "", sentence)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _extract_response_text(data: dict) -> str:
    texts = []
    for item in data.get("output", []):
        if item.get("type") != "message":
            continue
        if item.get("role") != "assistant":
            continue
        for block in item.get("content", []):
            if block.get("type") == "output_text":
                texts.append(block.get("text", ""))
    return "\n".join(texts)

def _call_gpt5(system_prompt, user_prompt):
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY is not set.")
    socket.setdefaulttimeout(REQUEST_TIMEOUT)
    if MAX_PROMPT_CHARS:
        if len(system_prompt) > MAX_PROMPT_CHARS:
            if DEBUG_LOG:
                print(f"[Prompt] system truncated {len(system_prompt)}->{MAX_PROMPT_CHARS}")
            system_prompt = system_prompt[:MAX_PROMPT_CHARS]
        if len(user_prompt) > MAX_PROMPT_CHARS:
            if DEBUG_LOG:
                print(f"[Prompt] user truncated {len(user_prompt)}->{MAX_PROMPT_CHARS}")
            user_prompt = user_prompt[:MAX_PROMPT_CHARS]

    payload = {
        "model": MODEL_NAME,
        "input": [
            {"role": "system", "content": [{"type": "input_text", "text": system_prompt}]},
            {"role": "user", "content": [{"type": "input_text", "text": user_prompt}]},
        ],
    }
    request = urllib.request.Request(
        "https://api.openai.com/v1/responses",
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        method="POST",
    )

    if DEBUG_LOG:
        print("[Call] gpt-5-nano request start")
        print(f"[Call] system_len={len(system_prompt)} user_len={len(user_prompt)}")

    last_error = None
    call_start = time.time()
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with urllib.request.urlopen(request, timeout=REQUEST_TIMEOUT) as response:
                data = json.loads(response.read().decode("utf-8"))
            content = _extract_response_text(data)
            text = content.strip()
            if text.startswith("```"):
                text = re.sub(r"^```[a-zA-Z0-9]*", "", text)
                text = re.sub(r"```$", "", text).strip()
            if DEBUG_LOG:
                print(f"[Call] gpt-5-nano response OK in {time.time() - call_start:.1f}s")
            return text
        except urllib.error.HTTPError as exc:
            body = exc.read().decode("utf-8", errors="replace")
            if exc.code in {408, 429, 500, 502, 503, 504} and attempt < MAX_RETRIES:
                if DEBUG_LOG:
                    print(f"[Retry] HTTP {exc.code} attempt {attempt}")
                time.sleep(BACKOFF_SECONDS * attempt)
                last_error = exc
                continue
            raise RuntimeError(f"OpenAI API error {exc.code}: {body}") from exc
        except (TimeoutError, urllib.error.URLError) as exc:
            if attempt < MAX_RETRIES:
                if DEBUG_LOG:
                    print(f"[Retry] Timeout/URLError attempt {attempt}")
                time.sleep(BACKOFF_SECONDS * attempt)
                last_error = exc
                continue
            raise RuntimeError("OpenAI API request timed out.") from exc

    raise RuntimeError("OpenAI API request failed.") from last_error

def _build_system_prompt(mode, rule):
    base = (
        "너는 CRM 메시지 문장을 편집하는 도우미다.\n"
        "반드시 한 문장만 출력하고, 따옴표/메타설명/번호/불릿을 붙이지 마라.\n"
        "문장은 자연스러운 한국어로 유지하되, 아래 규칙만 적용하라.\n"
    )
    if mode == "make_violation":
        rule_block = (
            "규칙: 지정된 규칙을 '오직 한 번' 위반하는 문장으로 바꿔라.\n"
            "다른 규칙 위반은 절대 만들지 마라.\n"
        )
    else:
        rule_block = (
            "규칙: 지정된 규칙 위반을 제거하고, 세 규칙 모두 위반하지 않는 문장으로 바꿔라.\n"
        )
    detail = (
        "- 페르소나 유출: 페르소나 이름(예: Budget_Seeker)을 문장에 직접 포함\n"
        "- 한/영 혼용: 한국어와 영어가 섞여 있음\n"
        "- 대괄호 표현: [ ] 사용\n"
    )
    return base + rule_block + detail

def _build_user_prompt(sentence, persona, rule, mode):
    rule_text = {
        "persona": "페르소나 유출",
        "mixing": "한/영 혼용",
        "brackets": "대괄호 표현",
    }[rule]
    if mode == "make_violation":
        instruction = (
            f"대상 규칙: {rule_text}\n"
            f"페르소나: {persona}\n"
            "요청: 아래 문장을 의미는 유지하면서 최소 수정으로 규칙을 위반하도록 바꿔라.\n"
            "조건: 다른 두 규칙은 위반하지 마라.\n"
        )
    else:
        instruction = (
            f"대상 규칙: {rule_text}\n"
            f"페르소나: {persona}\n"
            "요청: 아래 문장에서 규칙 위반을 제거하라.\n"
            "조건: 세 규칙 모두 위반하지 마라.\n"
        )
    return instruction + f"문장: {sentence}"

def _make_violation_fast(sentence, persona, rule):
    if rule == "persona" and persona:
        candidate = f"{persona}님, {sentence}"
        if _only_target_violation(candidate, persona, rule):
            return candidate
    if rule == "mixing":
        candidate = f"{sentence} skin"
        if _only_target_violation(candidate, persona, rule):
            return candidate
    if rule == "brackets":
        if " " in sentence:
            candidate = sentence.replace(" ", " [더블] ", 1)
        else:
            candidate = f"[더블] {sentence}"
        if _only_target_violation(candidate, persona, rule):
            return candidate
    return ""

def _fix_violation_fast(sentence, persona, rule):
    if rule == "persona" and persona:
        candidate = re.sub(re.escape(persona), "고객님", sentence, flags=re.IGNORECASE)
        if _no_violations(candidate, persona):
            return candidate
    if rule == "mixing":
        candidate = re.sub(r"[A-Za-z]+", "", sentence)
        candidate = re.sub(r"\s+", " ", candidate).strip()
        if _no_violations(candidate, persona):
            return candidate
    if rule == "brackets":
        candidate = sentence.replace("[", "").replace("]", "")
        if _no_violations(candidate, persona):
            return candidate
    return ""

def _generate_variant(sentence, persona, rule, mode, max_attempts=1):
    system_prompt = _build_system_prompt(mode, rule)
    user_prompt = _build_user_prompt(sentence, persona, rule, mode)
    last = ""
    for _ in range(max_attempts):
        text = _call_gpt5(system_prompt, user_prompt).strip()
        if not text:
            continue
        last = text
        if mode == "make_violation":
            if _only_target_violation(text, persona, rule):
                return text
        else:
            if _no_violations(text, persona):
                return text
    return last


def _fallback_make_violation(sentence, persona, rule):
    if rule == "persona" and persona:
        return f"{persona}님, {sentence}"
    if rule == "mixing":
        return f"{sentence} skin"
    if rule == "brackets":
        if " " in sentence:
            return sentence.replace(" ", " [더블] ", 1)
        return f"[더블] {sentence}"
    return sentence


def _fallback_fix(sentence, persona, rule):
    if rule == "persona" and persona:
        return re.sub(re.escape(persona), "고객님", sentence, flags=re.IGNORECASE)
    if rule == "mixing":
        text = re.sub(r"[A-Za-z]+", "", sentence)
        return re.sub(r"\s+", " ", text).strip()
    if rule == "brackets":
        return sentence.replace("[", "").replace("]", "")
    return sentence


os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)


total = 0
written = 0
skipped = 0

start_line_idx = 1
start_sent_idx = 0
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                last = json.loads(line)
                line_idx_val = last.get("line_idx")
                sent_idx_val = last.get("sent_idx")
                if isinstance(line_idx_val, int):
                    start_line_idx = line_idx_val
                    if isinstance(sent_idx_val, int):
                        start_sent_idx = sent_idx_val + 1
                    else:
                        applied_rule = "brackets"
                        applied_mode = "fix_violation"
                        start_sent_idx = 0
            except Exception:
                continue

if LOG_RESUME:
    print(f"Resume from line {start_line_idx}, sent {start_sent_idx}")

SAVE_EVERY = 20
buffer = []

def _flush_buffer():
    if not buffer:
        return
    with open(OUTPUT_PATH, "a", encoding="utf-8") as f_out:
        for record in buffer:
            f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
    buffer.clear()

stop_requested = False
with open(INPUT_PATH, "r", encoding="utf-8") as f_in:
    for line_idx, line in enumerate(f_in, 1):
        if MAX_INPUT_LINES and line_idx > MAX_INPUT_LINES:
            break
        if line_idx < start_line_idx:
            continue
        if not line.strip():
            continue
        row = json.loads(line)
        persona = _extract_persona(row.get("prompt", ""))
        fixed_prompt = _build_fixed_prompt(persona)
        chosen_raw = row.get("chosen", "")
        body = _extract_body(chosen_raw)
        sentences = _split_sentences(body)
        if MAX_SENTENCES_PER_ROW:
            sentences = sentences[:MAX_SENTENCES_PER_ROW]
        for sent_idx, sent in enumerate(sentences):
            if line_idx == start_line_idx and sent_idx < start_sent_idx:
                continue
            sent = sent.strip()
            if len(sent) < 10:
                continue
            total += 1
            violations = _all_violations(sent, persona)
            applied_rule = None
            applied_mode = None
            if any(violations.values()):
                if violations["persona"]:
                    applied_rule = "persona"
                    applied_mode = "fix_violation"
                    chosen = _fix_persona_local(sent, persona)
                    if not chosen:
                        chosen = sent
                    rejected = sent
                elif violations["mixing"]:
                    applied_rule = "mixing"
                    applied_mode = "fix_violation"
                    chosen = _generate_variant(sent, persona, "mixing", mode="fix_violation")
                    if not chosen:
                        chosen = sent
                    rejected = sent
                else:
                    applied_rule = "brackets"
                    applied_mode = "fix_violation"
                    chosen = _fix_brackets_local(sent)
                    if not chosen:
                        chosen = sent
                    rejected = sent

                if not _no_violations(chosen, persona):
                    skipped += 1
            else:
                chosen = sent
                rule = _pick_violation_rule()
                applied_rule = rule
                applied_mode = "make_violation"
                rejected = _generate_variant(sent, persona, rule, mode="make_violation")
                if not rejected:
                    rejected = _fallback_make_violation(sent, persona, rule)
                if not _only_target_violation(rejected, persona, rule):
                    skipped += 1

            violation_flags = _all_violations(sent, persona)
            violation_any = any(violation_flags.values())
            violation_type = applied_rule

            if LOG_RESULTS:
                print(f"[Meta] violation_any={violation_any} violation_type={violation_type}")
                print(f"[Chosen] {chosen}")
                print(f"[Rejected] {rejected}")
            record = {
                "prompt": fixed_prompt,
                "chosen": chosen,
                "rejected": rejected,
                "violation_any": violation_any,
                "violation_type": violation_type,
                "line_idx": line_idx,
                "sent_idx": sent_idx,
            }
            # print(f"{line_idx}처리 결과 : {record}")
            buffer.append(record)
            written += 1
            if PRINT_EVERY and written % PRINT_EVERY == 0:
                print(f"[Progress] line={line_idx} total={total} written={written} relaxed={skipped}")
            if MAX_RECORDS and written >= MAX_RECORDS:
                _flush_buffer()
                stop_requested = True
                break
            if len(buffer) >= SAVE_EVERY:
                _flush_buffer()

        if stop_requested:
            break


_flush_buffer()
print("Total sentences:", total)
print("Written records:", written)
print("Relaxed records:", skipped)
print("Output:", OUTPUT_PATH)


[Meta] violation_any=False violation_type=persona
[Chosen] 가격은 177,300원입니다.
[Rejected] Sensitive_Skin 고객님의 가격은 177,300원입니다.
[Meta] violation_any=False violation_type=mixing
[Chosen] 지금 바로 체험해 보세요.
[Rejected] 지금 바로 체험해 보세요, try 해보세요.
[Meta] violation_any=False violation_type=persona
[Chosen] 피부 자극 없이 보호를 돕는 저자극 크림 세트로, 임상 검증으로 안전성과 효과가 확인된 포뮬러를 제공합니다.
[Rejected] 피부 자극 없이 보호를 돕는 저자극 크림 세트로, 임상 검증으로 안전성과 효과가 확인된 Sensitive_Skin 포뮬러를 제공합니다.
[Meta] violation_any=False violation_type=persona
[Chosen] 피부 장벽 강화와 즉각적인 진정 효과를 기대하실 수 있으며, 건성 피부에는 깊은 보습과 회복을 한 번에 경험할 수 있습니다.
[Rejected] Sensitive_Skin으로 피부 장벽 강화와 즉각적인 진정 효과를 기대하실 수 있으며, 건성 피부에는 깊은 보습과 회복을 한 번에 경험할 수 있습니다.
[Meta] violation_any=False violation_type=persona
[Chosen] 대용량 구성으로 지속 사용이 가능하고, 리필 가능한 시스템으로 경제적입니다.
[Rejected] Sensitive_Skin 페르소나를 반영한 대용량 구성으로 지속 사용이 가능하고, 리필 가능한 시스템으로 경제적입니다.
[Meta] violation_any=False violation_type=mixing
[Chosen] 이번 런칭 이벤트로 브랜드별 첫 구매 시 사은품도 함께 제공되며, 신제품 런칭 특별 할인 혜택도 적용됩니다.
[Rejected] 이번 런칭 이벤트로 브랜드별 첫 구매 시